In [2]:
import scanpy as sc
adata = sc.read_h5ad("../data/obesity_challenge_2.h5ad")

In [14]:
adata.obs['gene']

cell
P1_AAACCCGCAACCCTTA-1     CEBPA+CEBPD
P1_AAACCCGCAACCTCAC-1          STAT5A
P1_AAACCCTGTATGGCAA-1          STAT5B
P1_AAACGAATCCTGGTCC-1           CEBPB
P1_AAACGACAGAACGGGC-1          STAT5B
                             ...     
P12_TGTGTTAGTCTGGTCA-1          KIF11
P12_TGTGTTAGTGATGCCC-1    KLF15+PPARG
P12_TGTGTTAGTTATCCAA-1             NC
P12_TGTGTTAGTTTAGCCG-1          SF3B1
P12_TGTGTTGAGTTACGCA-1          NR3C1
Name: gene, Length: 90815, dtype: category
Categories (237, object): ['CEBPA+CEBPA', 'CEBPA+CEBPA+CEBPB', 'CEBPA+CEBPA+NR3C1', 'CEBPA+CEBPA+POLR2D', ..., 'TCF7L2+TCF7L2', 'TCF7L2+ZBED3', 'ZBED3', 'ZBED3+ZBED3']

In [12]:
import pandas as pd

df = pd.DataFrame(
    adata.X.toarray(),
    index=adata.obs_names,
    columns=adata.var_names
)

df.head()

gene,MIR1302-2HG,FAM138A,OR4F5,AL627309.1,AL627309.3,AL627309.2,AL627309.5,AL627309.4,AP006222.2,AL732372.1,...,AC133551.1,AC136612.1,AC136616.1,AC136616.3,AC136616.2,AC141272.1,AC023491.2,AC007325.1,AC007325.4,AC007325.2
cell,,,,,,,,,,,,,,,,,,,,,
P1_AAACCCGCAACCCTTA-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
P1_AAACCCGCAACCTCAC-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.026242
P1_AAACCCTGTATGGCAA-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
P1_AAACGAATCCTGGTCC-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
P1_AAACGACAGAACGGGC-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000


In [15]:
import pandas as pd

# 讀取 txt (每行一個 perturbation)
with open("../data/predict_perturbations_2.txt") as f:
    predict_genes = [line.strip() for line in f if line.strip()]

predict_set = set(predict_genes)

print(f"# perturbations in txt: {len(predict_set)}")
print(list(predict_set)[:10])

# perturbations in txt: 62
['CREB1+CREB1', 'CEBPB+SREBF1', 'KLF15+PPARG2', 'FOXO1+MLXIPL', 'NC+PPARG2', 'NR3C1+TCF7L2', 'PPARG+SREBF1', 'FOXO1+PPARG2', 'CEBPD+NR3C1', 'STAT5B+ZBED3']


In [16]:
adata_genes = adata.obs["gene"].astype(str)

adata_set = set(adata_genes)

print(f"# perturbations in adata: {len(adata_set)}")
print(list(adata_set)[:10])

# perturbations in adata: 237
['NC+NR3C1+POLR2D', 'KLF15+KLF15', 'NR3C1+NR3C1+SF3B1', 'KLF15+NC+TCF7L2', 'SF3B1+STAT5A+STAT5A', 'KIF11+SF3B1', 'TCF7L2', 'CEBPD+PPARG', 'POLR2D+SREBF1', 'PPARG2+SF3B1']


In [17]:
common = adata_set.intersection(predict_set)

print(f"# overlap: {len(common)}")
print(sorted(common))

# overlap: 0
[]


In [21]:
import pandas as pd

# 轉成字串避免 category 問題
adata_genes = adata.obs["gene"].astype(str)

def split_genes(gene_iterable):
    genes = set()
    
    for g in gene_iterable:
        if g is None:
            continue
            
        parts = str(g).split("+")
        
        for p in parts:
            p = p.strip()
            
            if p != "":
                genes.add(p)
                
    return genes

adata_single_genes = split_genes(adata_genes)

print(f"# single genes in adata: {len(adata_single_genes)}")
print(sorted(list(adata_single_genes))[:30])

# single genes in adata: 19
['CEBPA', 'CEBPB', 'CEBPD', 'CREB1', 'FOXO1', 'KIF11', 'KLF15', 'MLXIPL', 'NC', 'NR3C1', 'POLR2D', 'PPARG', 'PPARG2', 'SF3B1', 'SREBF1', 'STAT5A', 'STAT5B', 'TCF7L2', 'ZBED3']


In [22]:
with open("../data/predict_perturbations_2.txt") as f:
    txt_genes = [line.strip() for line in f if line.strip()]

txt_single_genes = split_genes(txt_genes)

print(f"# single genes in txt: {len(txt_single_genes)}")
print(sorted(list(txt_single_genes)))

# single genes in txt: 19
['CEBPA', 'CEBPB', 'CEBPD', 'CREB1', 'FOXO1', 'KIF11', 'KLF15', 'MLXIPL', 'NC', 'NR3C1', 'POLR2D', 'PPARG', 'PPARG2', 'SF3B1', 'SREBF1', 'STAT5A', 'STAT5B', 'TCF7L2', 'ZBED3']


In [23]:
common_genes = adata_single_genes.intersection(txt_single_genes)

print(f"# overlapping genes: {len(common_genes)}")
print(sorted(common_genes))

# overlapping genes: 19
['CEBPA', 'CEBPB', 'CEBPD', 'CREB1', 'FOXO1', 'KIF11', 'KLF15', 'MLXIPL', 'NC', 'NR3C1', 'POLR2D', 'PPARG', 'PPARG2', 'SF3B1', 'SREBF1', 'STAT5A', 'STAT5B', 'TCF7L2', 'ZBED3']


In [24]:
common_genes = adata_single_genes & txt_single_genes
only_in_txt = txt_single_genes - adata_single_genes
only_in_adata = adata_single_genes - txt_single_genes

print("overlap:")
print(sorted(common_genes))

print("\nmissing in adata:")
print(sorted(only_in_txt))

print("\nextra genes in adata:")
print(sorted(list(only_in_adata))[:50])

overlap:
['CEBPA', 'CEBPB', 'CEBPD', 'CREB1', 'FOXO1', 'KIF11', 'KLF15', 'MLXIPL', 'NC', 'NR3C1', 'POLR2D', 'PPARG', 'PPARG2', 'SF3B1', 'SREBF1', 'STAT5A', 'STAT5B', 'TCF7L2', 'ZBED3']

missing in adata:
[]

extra genes in adata:
[]


In [ ]:
vc = adata.obs["gene"].value_counts()

num_gt4 = (vc > 4).sum()
num_lt4 = (vc < 4).sum()

print("數量 > 4 的 perturbations:", num_gt4)
print("數量 < 4 的 perturbations:", num_lt4)


In [ ]:
low_count = vc[vc < 4]

# 判斷是否為複合 perturbation
is_combo = low_count.index.str.contains(r"\+")

# 是否全部都是複合
all_combo = is_combo.all()

print("全部都是複合 perturbation:", all_combo)
num_combo = (low_count.index.str.contains(r"\+")).sum()
num_single = (~low_count.index.str.contains(r"\+")).sum()

print("複合 perturbations (<4 cells):", num_combo)
print("單一 gene perturbations (<4 cells):", num_single)

In [ ]:
vc = adata.obs["gene"].value_counts()

# 所有 perturbations
all_perts = vc.index.astype(str)

# 單基因 perturbations
single_perts = [p for p in all_perts if "+" not in p]

# 複合 perturbations
combo_perts = [p for p in all_perts if "+" in p]


# 拆開所有 combination 的 gene
combo_genes = set()

for pert in combo_perts:
    genes = pert.split("+")
    
    for g in genes:
        combo_genes.add(g.strip())


# 檢查每個單基因是否存在於 combination 中
single_not_in_combo = [
    g for g in single_perts
    if g not in combo_genes
]

print("單基因總數:", len(single_perts))
print("存在於 combination 中的單基因:", len(single_perts) - len(single_not_in_combo))
print("沒有出現在任何 combination 的單基因:", len(single_not_in_combo))

print("\n未出現在 combination 的單基因:")
print(sorted(single_not_in_combo))